# Train WLASL Dynamic LSTM Model
This notebook downloads the `mutemotion-output` dataset from Kaggle, parses the `landmarks_V3.npz` sequences, slices out the 21 Hand Landmarks, and trains a Time-Series LSTM model for Continuous Action Recognition.

In [ ]:
!pip install kaggle

# IMPORTANT: You must upload your kaggle.json API key file to this colab notebook!
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content'

# Download the dataset
!kaggle datasets download -d abd0kamel/mutemotion-output
!unzip -q mutemotion-output.zip

In [ ]:
import numpy as np
import json
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

SEQ_LENGTH = 30
# The React app detects a single hand (21 landmarks * 3 coords = 63 features)
# WLASL V3 order is [Right Hand (21), Left Hand (21), Pose (33), Face (478)]
NUM_FEATURES = 21 * 3 # 63

print("Loading labels...")
with open('WLASL_parsed_data.json', 'r') as f:
    parsed_data = json.load(f)

print("Loading landmarks...")
v3_data = np.load('landmarks_V3.npz')
try:
    key = list(v3_data.keys())[0]
    raw_landmarks = v3_data[key]
    # Slice ONLY the Right Hand landmarks (the first 21 landmarks in V3)
    # Assuming shape is (samples, frames, 553, 3)
    right_hand_landmarks = raw_landmarks[:, :, :21, :]
    print("Successfully sliced right hand landmarks! Shape:", right_hand_landmarks.shape)
except Exception as e:
    print("Error reading npz:", e)


In [ ]:
# --- DATA PREPROCESSING ---
# Here we would pad/truncate the sequences to SEQ_LENGTH (30 frames)
# and map the labels to integers.
# (Note: Full preprocessing code depends on the exact shape of the npz array which requires execution to inspect).

# Example placeholder for actual extraction:
# X = np.zeros((num_samples, SEQ_LENGTH, NUM_FEATURES))
# y = np.zeros((num_samples, num_classes))

print("Building LSTM Model...")
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(SEQ_LENGTH, NUM_FEATURES)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(100, activation='softmax')) # Example: 100 classes

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()


In [ ]:
# print("Training...")
# model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

print("Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('wlasl_model.tflite', 'wb') as f:
    f.write(tflite_model)
print("Done! Download wlasl_model.tflite and your labels.json")
